# Learned noise-aware attention — full experiment on Kaggle (T4 x2)

One notebook for the whole second contribution. **All checkpoints, logs and CSVs are saved into a state zip**
at the end of every session, so Kaggle's 12 h limit is handled by Save Version → attach that output to the
next version → Run all again. The notebook auto-detects the stage from what exists in the state:

1. **screen** (no `selection.json`): 3 short runs, control vs learned noise at 2 scales, 10k iters, parallel
   on the two T4s. Variant is picked by **validation PSNR only** (~1.5 h wall).
2. **full** (`selection.json` exists, 150k runs incomplete): matched control + selected variant, 150k iters
   from the official weights, one per GPU (~12.5 h wall = ~25 GPU-h). Uses a timeout; if the session ends,
   Save Version and re-attach to resume.
3. **eval** (both 150k runs complete): evaluate once on the held-out 100-image test set and write CSVs.

**Settings:** GPU T4 x2, Internet ON. Always use **Save & Run All**. After each stage, download
`learned_noise_state.zip` from the Output tab and attach it as input to the next version.


In [ ]:
import os, sys, subprocess, glob, json, zipfile
from pathlib import Path

os.chdir('/kaggle/working')
if not Path('btp').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Crazycraftie/low-light-enhancement-btp.git','btp'],check=True)
os.chdir('/kaggle/working/btp')
print('HEAD:', subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
if not Path('baselines/Retinexformer').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/caiyuanhao1998/Retinexformer.git','baselines/Retinexformer'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','einops','gdown','lmdb','yacs','addict','future','natsort','yapf','thop','timm','lpips','scikit-image'],check=True)

import torch
assert torch.cuda.is_available() and torch.cuda.device_count()>=2, 'Enable GPU T4 x2'
NGPU=torch.cuda.device_count()
OPT='baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'
RUNS=Path('runs/learned_noise')
SEL=Path('selection.json')
print('GPUs:',[torch.cuda.get_device_name(i) for i in range(NGPU)])


## Restore previous session state (if attached)


In [ ]:
zips = glob.glob('/kaggle/input/**/learned_noise_state*.zip', recursive=True)
if zips:
    archive = max(zips, key=os.path.getsize)
    with zipfile.ZipFile(archive) as zf: zf.extractall('/kaggle/working/btp')
    print('Restored state from', archive)
else:
    print('No prior state attached - starting fresh.')


def save_state():
    path='/kaggle/working/learned_noise_state.zip'
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as zf:
        for root in (RUNS, SEL):
            if root.exists():
                if root.is_file(): zf.write(root,root.name)
                else:
                    for p in root.rglob('*'):
                        if p.is_file(): zf.write(p,p.as_posix())
        if Path('results/results.csv').exists(): zf.write('results/results.csv','results.csv')
        if Path('results/dark_region.csv').exists(): zf.write('results/dark_region.csv','dark_region.csv')
    print('Saved',path)

# Auto-detect stage from restored state.
if not SEL.exists(): STAGE='screen'
elif not all((RUNS/'final'/n/'metrics.json').exists() for n in ('control','learned_noise')): STAGE='full'
else: STAGE='eval'
print('STAGE:',STAGE)


## Data + official LOL-v2-syn weights (cached in the working dir)


In [ ]:
def gdl(fid,out,tries=4):
    out=Path(out); out.parent.mkdir(parents=True,exist_ok=True)
    for _ in range(tries):
        if subprocess.run(['gdown','-q',fid,'-O',str(out)]).returncode==0 and out.exists(): return
    raise RuntimeError('download failed '+str(out))

DATA=Path('data/LOLv2/Synthetic')
if not (DATA/'Train/Low').is_dir():
    attached=glob.glob('/kaggle/input/**/LOLv2/Synthetic/Train/Low',recursive=True)
    if attached:
        src=Path(attached[0]).parents[2]; Path('data').mkdir(exist_ok=True)
        if not Path('data/LOLv2').exists(): Path('data/LOLv2').symlink_to(src,target_is_directory=True)
    else:
        gdl('1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U','/kaggle/tmp/lolv2.zip')
        subprocess.run(['unzip','-q','-o','/kaggle/tmp/lolv2.zip','-d','data'],check=True)
assert len(list((DATA/'Train/Low').glob('*')))==900
assert len(list((DATA/'Test/Low').glob('*')))==100

WDIR=Path('baselines/Retinexformer/pretrained_weights')
W=WDIR/'LOL_v2_synthetic.pth'
if not W.exists():
    c=list(WDIR.glob('**/LOL_v2_synthetic.pth'))
    if not c:
        subprocess.run(['gdown','--folder','1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV','-O','/kaggle/tmp/rf_weights'],check=True)
        c=list(Path('/kaggle/tmp/rf_weights').glob('**/LOL_v2_synthetic.pth'))
    assert c,'weights not found'
    W.parent.mkdir(parents=True,exist_ok=True)
    import shutil; shutil.copyfile(str(c[0]),str(W))
W=str(W); print('data+weights ready:',W)


## Training helper: parallel tasks, resumable, tolerant of timeouts


In [ ]:
def run_pair(tasks, timeout):
    """tasks: list of dicts {out, scale, iters, lr}. Runs them one per GPU."""
    launched=[]
    for gpu,t in enumerate(tasks):
        out=Path(t['out']); out.mkdir(parents=True,exist_ok=True)
        if (out/'metrics.json').exists():
            print('already complete:',out); continue
        cmd=['timeout',str(timeout),sys.executable,'scripts/finetune.py',
             '--loss','l1','--iters',str(t['iters']),'--lr',str(t['lr']),
             '--batch','8','--crop','128','--seed','42','--pretrained',W,'--opt',OPT,
             '--repo','baselines/Retinexformer','--data',str(DATA/'Train'),'--device','cuda',
             '--val-fraction','0.1','--val-seed','2026','--val-every',str(t['val_every']),
             '--checkpoint-every','5000','--log-every','500','--out',str(out)]
        if t['scale'] is not None: cmd += ['--learned-noise','--noise-scale',str(t['scale'])]
        if (out/'latest_state.pth').exists(): cmd += ['--resume']
        env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu)
        log=(out/'run.log').open('a')
        launched.append((t,subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env),log))
    for t,proc,log in launched:
        rc=proc.wait(); log.close(); print(t['out'],'exit',rc)
        if rc not in (0,124): raise RuntimeError((Path(t['out'])/'run.log').read_text()[-4000:])


## Stage 1 — screening on validation only


In [ ]:
if STAGE=='screen':
    tasks=[]
    for name,scale in (('control',None),('learned_s0.5',0.5),('learned_s1.0',1.0)):
        tasks.append({'out':str(RUNS/'screen'/name),'scale':scale,'iters':10000,'lr':2e-5,'val_every':1000})
    run_pair(tasks, timeout=5000)
    save_state()   # save before reading metrics
    scores={}
    for name,_ in (('control',None),('learned_s0.5',0.5),('learned_s1.0',1.0)):
        m=RUNS/'screen'/name/'metrics.json'
        if m.exists(): scores[name]=json.loads(m.read_text())['best_val_psnr']
    print('validation PSNR:',scores)
    best=max(scores,key=scores.get) if scores else None
    SEL.write_text(json.dumps({'scores':scores,'best':best},indent=2))
    save_state()
    print('SELECTED ON VALIDATION:',best)
    if best not in ('learned_s0.5','learned_s1.0'):
        print('Control won validation - no evidence of a gain. The final paper stands as-is.')


## Stage 2 — matched 150k runs


In [ ]:
if STAGE=='full':
    sel=json.loads(SEL.read_text())
    best=sel['best']
    if best not in ('learned_s0.5','learned_s1.0'):
        raise RuntimeError('Control won validation - stopping before the expensive runs.')
    SCALE=0.5 if best=='learned_s0.5' else 1.0
    tasks=[{'out':str(RUNS/'final'/'control'),'scale':None,'iters':150000,'lr':2e-4,'val_every':5000},
           {'out':str(RUNS/'final'/'learned_noise'),'scale':SCALE,'iters':150000,'lr':2e-4,'val_every':5000}]
    try:
        run_pair(tasks, timeout=39000)   # 10.8 h, under Kaggle's 12 h session
    finally:
        save_state()
    print('final runs done:',[(n,(RUNS/'final'/n/'metrics.json').exists()) for n in ('control','learned_noise')])


## Stage 3 — one held-out test evaluation


In [ ]:
if STAGE=='eval':
    SCALE=0.5 if json.loads(SEL.read_text())['best']=='learned_s0.5' else 1.0
    def infer(weights,inp,out,method,scale=None):
        cmd=[sys.executable,'scripts/retinexformer_infer.py','--repo','baselines/Retinexformer',
             '--opt',OPT,'--weights',str(weights),'--input',str(inp),'--output',str(out),
             '--method',method,'--device','0','--no-speed-log']
        if scale is not None: cmd += ['--learned-noise','--noise-scale',str(scale)]
        subprocess.run(cmd,check=True)
    for name,scale in (('control',None),('learned_noise',SCALE)):
        run=RUNS/'final'/name
        w=run/'best.pth'
        if not w.exists(): w=run/'final.pth'
        method='LearnedNoise-control' if scale is None else f'LearnedNoise-a{SCALE}'
        out=Path('results/learned_noise')/method/'LOLv2-syn'
        infer(w,DATA/'Test/Low',out,method,scale)
        for script,extra in (
            ('scripts/evaluate.py',['--gt',str(DATA/'Test/Normal')]),
            ('scripts/eval_dark_regions.py',['--gt',str(DATA/'Test/Normal'),'--low',str(DATA/'Test/Low')])):
            subprocess.run([sys.executable,script,'--pred',str(out),'--method',method,
                            '--dataset','LOLv2-syn']+extra,check=True)
    save_state()
    print('TEST RESULTS:')
    print('\n'.join(l for l in Path('results/results.csv').read_text().splitlines() if 'LearnedNoise' in l))


## Done

Download `learned_noise_state.zip` from the Output tab. If the stage was `screen` or `full`, attach it to the
next version and Run all again. If the stage was `eval`, send me the zip and I'll run the paired statistics
and write up the result (improvement or third negative result).
